## 1. Setup

First, let's set up the paths and imports.

In [6]:
import subprocess
import json
import os
import tempfile
import numpy as np

# Path to the executable (adjust if needed)
EXE_PATH = os.path.join(os.getcwd(), 'classify_behavior.exe')

# Verify the executable exists
if os.path.exists(EXE_PATH):
    print(f"✓ Found executable: {EXE_PATH}")
else:
    print(f"✗ Executable not found at: {EXE_PATH}")
    print("  Run 'python build.py' first to create the executable.")

✓ Found executable: c:\Users\alexa\Documents\GitHub\sd-ai\evals\model_output_evaluation\behavioral_evaluation_using_ists\dist_examples\classify_behavior.exe


## 2. Prepare Input Data

The executable accepts CSV or JSON files. Here's the expected format:

### CSV Format
```csv
value
10.5
12.3
15.1
...
```

### JSON Format
```json
[10.5, 12.3, 15.1, ...]
```

Let's create some synthetic time series data:

In [7]:
# Generate synthetic exponential growth data
np.random.seed(42)
t = np.linspace(0, 10, 100)
exponential_data = 5 * np.exp(0.3 * t) + np.random.normal(0, 2, len(t))

# Generate oscillation data
oscillation_data = 50 + 20 * np.sin(2 * np.pi * 0.5 * t) + np.random.normal(0, 1, len(t))

print(f"Generated {len(exponential_data)} data points for each pattern")
print(f"Exponential growth range: {exponential_data.min():.2f} to {exponential_data.max():.2f}")
print(f"Oscillation range: {oscillation_data.min():.2f} to {oscillation_data.max():.2f}")

Generated 100 data points for each pattern
Exponential growth range: 3.59 to 99.96
Oscillation range: 29.50 to 72.13


## 3. Save Data to File

Save the data in a format the executable can read.

In [8]:
# Create a temporary directory for our test files
temp_dir = tempfile.mkdtemp(prefix="classify_behavior_demo_")

# Save as CSV
csv_path = os.path.join(temp_dir, "exponential_growth.csv")
with open(csv_path, 'w') as f:
    f.write("value\n")  # Header
    for value in exponential_data:
        f.write(f"{value}\n")

print(f"✓ Saved CSV file: {csv_path}")

# Save as JSON
json_path = os.path.join(temp_dir, "oscillation.json")
with open(json_path, 'w') as f:
    json.dump(oscillation_data.tolist(), f)

print(f"✓ Saved JSON file: {json_path}")

✓ Saved CSV file: C:\Users\alexa\AppData\Local\Temp\classify_behavior_demo_a6z56tfu\exponential_growth.csv
✓ Saved JSON file: C:\Users\alexa\AppData\Local\Temp\classify_behavior_demo_a6z56tfu\oscillation.json


## 4. Call the Executable

Now let's call the executable and get the classification results.

### Basic Command Syntax
```
classify_behavior.exe <input_file> [options]

Options:
  --format json    Output as JSON (default: text)
  --format csv     Output as CSV
  --hypothesis X   Test if data matches pattern X
  --top N          Show top N matches (default: 3)
  --list-patterns  List all available pattern codes
```

In [9]:
def classify_timeseries(file_path, exe_path=EXE_PATH, output_format='json'):
    """
    Call classify_behavior.exe to classify a time series.
    
    Args:
        file_path: Path to CSV or JSON file with time series data
        exe_path: Path to the executable
        output_format: 'json', 'text', or 'csv'
    
    Returns:
        dict if format='json', else str
    """
    cmd = [exe_path, file_path, '--format', output_format]
    
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=60
    )
    
    if result.returncode != 0:
        raise RuntimeError(f"Classification failed: {result.stderr}")
    
    if output_format == 'json':
        return json.loads(result.stdout)
    else:
        return result.stdout

print("✓ Helper function defined")

✓ Helper function defined


### Classify Exponential Growth Data (CSV)

In [10]:
# Classify the exponential growth data
result_exp = classify_timeseries(csv_path)

print("Classification Result (Exponential Growth):")
print(json.dumps(result_exp, indent=2))

Classification Result (Exponential Growth):
{
  "class_id": 6,
  "class_name": "pexgr",
  "class_description": "Positive Exponential Growth",
  "likelihood": -2.7116359063873423,
  "is_weak_match": true,
  "top_matches": [
    {
      "class_id": 6,
      "class_name": "pexgr",
      "class_description": "Positive Exponential Growth",
      "likelihood": -2.7116359063873423
    },
    {
      "class_id": 11,
      "class_name": "d1peg",
      "class_description": "Linear Decay to Positive Exp Growth",
      "likelihood": -5.417806586336438
    },
    {
      "class_id": 23,
      "class_name": "oscgr",
      "class_description": "Oscillation with Growth",
      "likelihood": -7.2421198310139605
    }
  ]
}


### Classify Oscillation Data (JSON)

In [11]:
# Classify the oscillation data
result_osc = classify_timeseries(json_path)

print("Classification Result (Oscillation):")
print(json.dumps(result_osc, indent=2))

Classification Result (Oscillation):
{
  "class_id": 22,
  "class_name": "oscct",
  "class_description": "Oscillation Constant",
  "likelihood": -1.6378818132687585,
  "is_weak_match": false,
  "top_matches": [
    {
      "class_id": 22,
      "class_name": "oscct",
      "class_description": "Oscillation Constant",
      "likelihood": -1.6378818132687585
    },
    {
      "class_id": 24,
      "class_name": "oscdc",
      "class_description": "Oscillation with Decay",
      "likelihood": -7.2421198310139605
    },
    {
      "class_id": 23,
      "class_name": "oscgr",
      "class_description": "Oscillation with Growth",
      "likelihood": -10.0
    }
  ]
}


## 5. Parse and Display Results

Let's create a nice display of the classification results.

In [12]:
def display_classification(result, name="Time Series"):
    """
    Display classification results in a readable format.
    """
    print("=" * 50)
    print(f"Classification: {name}")
    print("=" * 50)
    print(f"  Pattern Code:    {result['class_name']}")
    print(f"  Pattern ID:      {result['class_id']}")
    print(f"  Description:     {result['class_description']}")
    print(f"  Likelihood:      {result['likelihood']:.4f}")
    print(f"  Weak Match:      {'Yes' if result['is_weak_match'] else 'No'}")
    print()
    print("  Top Matches:")
    for i, match in enumerate(result['top_matches'], 1):
        print(f"    {i}. {match['class_name']:<8} ({match['class_description']})")
        print(f"       Likelihood: {match['likelihood']:.4f}")
    print("=" * 50)

# Display both results
display_classification(result_exp, "Exponential Growth Data")
print()
display_classification(result_osc, "Oscillation Data")

Classification: Exponential Growth Data
  Pattern Code:    pexgr
  Pattern ID:      6
  Description:     Positive Exponential Growth
  Likelihood:      -2.7116
  Weak Match:      Yes

  Top Matches:
    1. pexgr    (Positive Exponential Growth)
       Likelihood: -2.7116
    2. d1peg    (Linear Decay to Positive Exp Growth)
       Likelihood: -5.4178
    3. oscgr    (Oscillation with Growth)
       Likelihood: -7.2421

Classification: Oscillation Data
  Pattern Code:    oscct
  Pattern ID:      22
  Description:     Oscillation Constant
  Likelihood:      -1.6379
  Weak Match:      No

  Top Matches:
    1. oscct    (Oscillation Constant)
       Likelihood: -1.6379
    2. oscdc    (Oscillation with Decay)
       Likelihood: -7.2421
    3. oscgr    (Oscillation with Growth)
       Likelihood: -10.0000


## 6. Using Text Output Format

You can also get human-readable text output:

In [13]:
# Get text format output
text_result = classify_timeseries(csv_path, output_format='text')
print(text_result)

BEHAVIOR CLASSIFICATION RESULT
Class ID:          6
Class Name:        pexgr
Description:       Positive Exponential Growth
Likelihood:        -2.7116
Weak Match:        Yes

Top Matches:
  1. pexgr (Positive Exponential Growth) - -2.7116
  2. d1peg (Linear Decay to Positive Exp Growth) - -5.4178
  3. oscgr (Oscillation with Growth) - -7.2421



## 7. Testing a Hypothesis

You can test if your data matches a specific pattern:

In [14]:
def test_hypothesis(file_path, hypothesis_code, exe_path=EXE_PATH):
    """
    Test if time series matches a hypothesized pattern.
    
    Args:
        file_path: Path to data file
        hypothesis_code: Pattern code to test (e.g., 'pexgr', 'oscct')
    
    Returns:
        dict with hypothesis test results
    """
    cmd = [exe_path, file_path, '--hypothesis', hypothesis_code, '--format', 'json']
    
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    
    if result.returncode != 0:
        raise RuntimeError(f"Hypothesis test failed: {result.stderr}")
    
    return json.loads(result.stdout)

# Test correct hypothesis
print("Testing hypothesis: Is oscillation data 'oscct' (Oscillation Constant)?")
hypothesis_result = test_hypothesis(json_path, 'oscct')
print(json.dumps(hypothesis_result, indent=2))

print()

# Test incorrect hypothesis
print("Testing hypothesis: Is oscillation data 'pexgr' (Exponential Growth)?")
hypothesis_result2 = test_hypothesis(json_path, 'pexgr')
print(json.dumps(hypothesis_result2, indent=2))

Testing hypothesis: Is oscillation data 'oscct' (Oscillation Constant)?
{
  "hypothesis": "oscct",
  "confirmed": true,
  "is_weak_match": false
}

Testing hypothesis: Is oscillation data 'pexgr' (Exponential Growth)?
{
  "hypothesis": "pexgr",
  "confirmed": false,
  "is_weak_match": false
}


## 8. List Available Patterns

To see all available pattern codes:

In [15]:
# List all available patterns
result = subprocess.run(
    [EXE_PATH, '--list-patterns'],
    capture_output=True,
    text=True
)
print(result.stdout)


Available Behavior Pattern Codes:
ID   Code     Description                             
------------------------------------------------------------
0    zero0    Zero/No Data                            
1    const    Constant/Stasis                         
2    plinr    Positive Linear Growth                  
3    nlinr    Negative Linear (Decay)                 
4    nexgr    Negative Exponential Growth             
5    sshgr    S-Shaped Growth                         
6    pexgr    Positive Exponential Growth             
7    gr1da    Linear Growth with Decay (A)            
8    gr1db    Linear Growth with Decay (B)            
9    gr2da    Exponential Growth with Decay (A)       
10   gr2db    Exponential Growth with Decay (B)       
11   d1peg    Linear Decay to Positive Exp Growth     
12   d2peg    Exp Decay to Positive Exp Growth        
13   nexdc    Negative Exponential Decay              
14   sshdc    S-Shaped Decay                          
15   pexdc    Positive E

## 9. Cleanup

Remove temporary files.

In [16]:
import shutil

# Clean up temporary files
shutil.rmtree(temp_dir)
print(f"✓ Cleaned up temporary directory: {temp_dir}")

✓ Cleaned up temporary directory: C:\Users\alexa\AppData\Local\Temp\classify_behavior_demo_a6z56tfu


## Summary

### Input Format
- **CSV**: Single column with header `value`, one number per line
- **JSON**: Array of numbers `[1.0, 2.0, 3.0, ...]`

### Command Examples
```bash
# Basic classification
classify_behavior.exe data.csv

# JSON output
classify_behavior.exe data.csv --format json

# Test a hypothesis
classify_behavior.exe data.csv --hypothesis pexgr

# List patterns
classify_behavior.exe --list-patterns
```

### JSON Response Structure
```json
{
  "class_id": 6,
  "class_name": "pexgr",
  "class_description": "Positive Exponential Growth",
  "likelihood": -5.1234,
  "is_weak_match": false,
  "top_matches": [...]
}
```